# lcz4r_python — Berlin local climate analysis

Tutorial using `lcz_data_berlin.csv` (23 stations, hourly air temperature, 2019–2020).
Run cells top-to-bottom. The LCZ map for Berlin is downloaded in §1 and reused in all sections.

In [2]:
import sys, os
sys.path.insert(0, os.getcwd())

import pandas as pd

from lcz_get_map_euro import lcz_get_map_euro
from lcz_ts import lcz_ts
from lcz_anomaly import lcz_anomaly
from lcz_uhi_intensity import lcz_uhi_intensity
from lcz_interp_map import lcz_interp_map
from lcz_interp_eval import lcz_interp_eval
from lcz_anomaly_map import lcz_anomaly_map
from lcz_plot_interp import lcz_plot_interp
from lcz_get_ucp import lcz_get_ucp

## 1 — LCZ map for Berlin

Downloads the European LCZ map clipped to Berlin and caches it locally.
Subsequent runs load from cache in milliseconds.

In [3]:
map_path = lcz_get_map_euro(city="Berlin", isave_map=False)
print("map_path:", map_path)

10:58:04 - _lcz_map_engine - INFO - Streaming COG and clipping to geometry...
10:58:05 - rasterio._env - WARNING - CPLE_AppDefined in HTTP response code on https://zenodo.org/records/10835692/files/EU_LCZ_map.tiff?download=1.xml: 403


RetryError: RetryError[<Future at 0x18a62a550 state=finished raised RasterioIOError>]

## 2 — Load Berlin meteorological data

`lcz_data_berlin.csv` contains hourly air temperature (°C) from 23 stations across
Berlin and surroundings (January 2019 – November 2020).

Column names used by all functions:
| CSV column | Role | Auto-detected? |
|-----------|------|----------------|
| `date` | timestamp | ✓ (regex) |
| `Latitude` | latitude | ✓ (regex) |
| `Longitude` | longitude | ✓ (regex) |
| `airT` | temperature variable | explicit: `var="airT"` |
| `station` | station ID | explicit: `station_id="station"` |

In [3]:
df = pd.read_csv("lcz_data_berlin.csv")
print(f"{len(df):,} rows — {df['station'].nunique()} stations — {df['date'].min()} to {df['date'].max()}")
df.head(3)

386,354 rows — 23 stations — 2019-01-01 00:00:00 to 2020-11-30 23:00:00


,Unnamed: 0,date,station,airT,Latitude,Longitude
0,806404,2019-01-01 00:00:00,airportthf,8.50000,52.467500,13.402100
1,806405,2019-01-01 00:00:00,airporttxl,7.90000,52.564400,13.308800
2,806406,2019-01-01 00:00:00,albrecht,8.04725,52.444594,13.348607


## 3 — Time series  `lcz_ts`

Plots LCZ-stratified time series. Each station line is coloured by its assigned
LCZ class.  
We filter to **summer 2019** (June–August) and average to **daily** resolution
so the chart stays readable.

In [6]:
# Basic line — all stations, summer 2019, daily averages
fig_ts = lcz_ts(
    map_path, df,
    var="airT", station_id="station",
    time_freq="day",
    plot_type="basic_line",
    palette="VanGogh2",
    year=2019, month=[6, 7, 8],
    title="Berlin — Daily Air Temperature (Summer 2019)",
    lang="en",
)
fig_ts.show()

In [7]:
# Heatmap variant — one row per station, time on x-axis
fig_heat = lcz_ts(
    map_path, df,
    var="airT", station_id="station",
    time_freq="day",
    plot_type="heatmap",
    facet_plot="station",
    year=2019, month=[6, 7, 8],
    title="Berlin — Daily Temperature Heatmap (Summer 2019)",
    lang="en",
)
fig_heat.show()

In [9]:
# Portuguese labels — same data, different lang=
fig_ts_pt = lcz_ts(
    map_path, df,
    var="airT", station_id="station",
    time_freq="day",
    year=2019, month=[6, 7, 8],
    lang="pt",
)
fig_ts_pt.show()

### 3b — `by=` temporal splitting

The `by=` argument facets the time series into sub-panels by a temporal grouping key.
Supported values: `"year"`, `"season"`, `"seasonyear"`, `"month"`, `"monthyear"`,
`"weekday"`, `"weekend"`, `"site"`, `"daylight"`, `"dst"`.

**Gap-line suppression is automatic**: when data within a panel is non-contiguous
(e.g. all Mondays, or all winter months), the connecting line is broken at large
time gaps — no spurious diagonals.

In [10]:
# by="season" — four sub-panels, one per meteorological season (DJF/MAM/JJA/SON)
fig_season = lcz_ts(
    map_path, df,
    var="airT", station_id="station",
    time_freq="day",
    by="season",
    palette="VanGogh2",
    year=2019,
    title="Berlin 2019 — Air Temperature by Season",
    lang="en",
)
fig_season.show()

In [11]:
# by="weekend" — weekday vs weekend patterns over summer
fig_weekend = lcz_ts(
    map_path, df,
    var="airT", station_id="station",
    time_freq="hour",
    by="weekend",
    palette="Cassatt1",
    year=2019, month=[6, 7, 8],
    title="Berlin Summer 2019 — Weekday vs Weekend",
    lang="en",
)
fig_weekend.show()

### 3c — `by="daylight"`: unified day/night chart

When `by="daylight"`, **no subplot split is created**. Instead, the full time series
is shown in a single chart with **amber shading** marking each daytime window.

Daytime is computed from the **NOAA/Spencer solar elevation formula** (UTC timestamps
assumed), valid globally — including polar regions (polar day → fully shaded,
polar night → no shading) and UTC-east cities like Sydney (split-day UTC intervals
handled automatically).

In [12]:
# by="daylight" — full time series with amber daytime shading (one week, hourly)
fig_daylight = lcz_ts(
    map_path, df,
    var="airT", station_id="station",
    time_freq="hour",
    by="daylight",
    palette="VanGogh2",
    start="2019-07-01", end="2019-07-14",
    title="Berlin — Air Temperature with Daytime Periods (July 2019)",
    caption="Amber shading = daytime (UTC, NOAA solar elevation formula)",
    lang="en",
)
fig_daylight.show()

## 4 — Temperature anomaly  `lcz_anomaly`

For each station, computes the mean temperature minus the overall spatial mean
across all stations. Positive values = warmer than average (urban heat effect).

Bar colour = LCZ class of each station.

In [8]:
# Default plot: diverging horizontal bar (positive = warmer than average)
fig_anom = lcz_anomaly(
    map_path, df,
    var="airT", station_id="station",
    plot_type="diverging_bar",
    year=2019, month=[6, 7, 8],
    lang="en",
    caption="Berlin, June–August 2019",
)
fig_anom.show()

In [9]:
# plot_type="bar" — vertical bar, red = warm / blue = cool, colorblind palette
fig_anom_bar = lcz_anomaly(
    map_path, df,
    var="airT", station_id="station",
    plot_type="bar",
    year=2019, month=[6, 7, 8],
    inclusive=True,
    lang="en",
)
fig_anom_bar.show()

In [10]:
# Return the raw anomaly table instead of a figure
anom_df = lcz_anomaly(
    map_path, df,
    var="airT", station_id="station",
    year=2019,
    iplot=False,
)
anom_df.sort("anomaly", descending=True)

station,mean_value,lcz,reference_value,anomaly
str,f64,i32,f64,f64
"""dessauer (8)""",12.446683,8,11.460761,0.99
"""bamberger (2)""",12.279626,2,11.460761,0.82
"""airportthf (8)""",11.814532,8,11.460761,0.35
"""marzahn (6)""",11.723884,6,11.460761,0.26
"""airporttxl (14)""",11.679153,14,11.460761,0.22
…,…,…,…,…
"""dahlem (6)""",11.18892,6,11.460761,-0.27
"""kaniswall (12)""",11.071863,12,11.460761,-0.39
"""dahlemerfeld (11)""",10.945419,11,11.460761,-0.52


In [11]:
# plot_type="dot" — Cleveland dot plot: mean temperature vs reference, connected lines
fig_anom_dot = lcz_anomaly(
    map_path, df,
    var="airT", station_id="station",
    plot_type="dot",
    year=2019, month=[6, 7, 8],
    lang="en",
    caption="Dot = station mean; grey dot = overall mean; line = anomaly",
)
fig_anom_dot.show()

In [12]:
# plot_type="lollipop" — stem + dot, minimalist anomaly magnitude
fig_anom_lol = lcz_anomaly(
    map_path, df,
    var="airT", station_id="station",
    plot_type="lollipop",
    year=2019, month=[6, 7, 8],
    lang="zh",
)
fig_anom_lol.show()

In [13]:
# plot_type="scatter" — LCZ class (x) vs mean temperature (y); dot area ∝ |anomaly|
fig_anom_scat = lcz_anomaly(
    map_path, df,
    var="airT", station_id="station",
    plot_type="scatter",
    year=2019, month=[6, 7, 8],
    lang="en",
    caption="Dot area encodes |anomaly|. Useful for cross-LCZ comparison.",
)
fig_anom_scat.show()

### 4b — `by=` temporal splitting for anomaly

Pass `by=` to `lcz_anomaly` to compute anomalies **within each temporal group**.
Each sub-panel shows station deviations relative to that group's mean — useful for
comparing how the urban heat effect varies by season or time of day.

In [14]:
# by="season" — within-season anomaly, one diverging-bar panel per season
fig_anom_season = lcz_anomaly(
    map_path, df,
    var="airT", station_id="station",
    plot_type="diverging_bar",
    by="season",
    year=2019,
    lang="en",
    caption="Anomaly relative to within-season mean",
)
fig_anom_season.show()

## 5 — Urban Heat Island intensity  `lcz_uhi_intensity`

UHI intensity = mean temperature of **urban** stations (LCZ 1–10) minus
mean temperature of **rural** stations (LCZ 11–16), averaged per time step.

`method="LCZ"` assigns each station automatically from the raster.

In [15]:
# Single-axis plot: UHI time series only (daily averages, full 2019)
fig_uhi = lcz_uhi_intensity(
    map_path, df,
    var="airT", station_id="station",
    time_freq="day",
    method="LCZ",
    group=False,
    year=2019,
    lang="en",
    caption="Berlin 2019 — daily UHI intensity",
)
fig_uhi.show()

In [16]:
# Dual-axis plot: urban & rural temperatures + UHI overlay
fig_uhi_dual = lcz_uhi_intensity(
    map_path, df,
    var="airT", station_id="station",
    time_freq="day",
    method="LCZ",
    group=True,
    year=2019,
    lang="en",
    caption="Berlin 2019 — urban vs. rural temperatures",
)
fig_uhi_dual.show()

### 5b — UHI with `by=` temporal splitting

Pass `by=` to `lcz_uhi_intensity` to facet the UHI signal by season, daylight period,
weekday, etc. Each sub-panel shows the UHI time series for that temporal group.
`group=True` adds urban/rural temperature traces on a secondary axis.

In [17]:
# by="season" — UHI faceted by season (four sub-panels, daily resolution)
fig_uhi_season = lcz_uhi_intensity(
    map_path, df,
    var="airT", station_id="station",
    time_freq="day",
    by="season",
    method="LCZ",
    group=False,
    year=2019,
    lang="en",
    caption="Berlin 2019 — seasonal UHI intensity",
)
fig_uhi_season.show()

In [5]:
# by="daylight" — daytime vs nighttime UHI (hourly, summer 2019)
fig_uhi_daylight = lcz_uhi_intensity(
    map_path, df,
    var="airT", station_id="station",
    time_freq="hour",
    by="daylight",
    method="LCZ",
    group=False,
    year=2019, month=[6, 7, 8],
    lang="en",
    caption="Daytime UHI vs nighttime UHI — summer 2019",
)
fig_uhi_daylight.show()

In [6]:
# Portuguese labels — dual-axis (urban/rural temps + UHI)
fig_uhi_pt = lcz_uhi_intensity(
    map_path, df,
    var="airT", station_id="station",
    time_freq="day",
    year=2019,
    method="LCZ",
    group=True,
    lang="pt",
)
fig_uhi_pt.show()

In [23]:
# Return wide-format table: date | urban | rural | uhi
uhi_df = lcz_uhi_intensity(
    map_path, df,
    var="airT", station_id="station",
    time_freq="month",
    iplot=False,
)
uhi_df

date,rural,urban,uhi
datetime[μs],f64,f64,f64
2019-01-01 00:00:00,1.290555,1.581073,0.29
2019-02-01 00:00:00,4.281089,4.785296,0.5
2019-03-01 00:00:00,6.897727,7.21954,0.32
2019-04-01 00:00:00,11.081653,11.568619,0.49
2019-05-01 00:00:00,12.575486,13.141568,0.57
…,…,…,…
2020-07-01 00:00:00,18.157548,18.810565,0.65
2020-08-01 00:00:00,21.347164,21.868713,0.52
2020-09-01 00:00:00,15.226967,15.918168,0.69


## 6 — Spatial interpolation  `lcz_interp_map`

Kriging-based spatial interpolation of air temperature onto a regular grid
covering the Berlin LCZ map extent.  
Output: a multi-band GeoTIFF (one band per time step).

> **Requires**: `pykrige` and `pyproj`.  
> We restrict to **one week** of daily data to keep runtime reasonable.

In [1]:
interp_path = lcz_interp_map(
    map_path, df,
    var="airT", station_id="station",
    sp_res=500.0,          # 500 m grid — fast for demo
    tp_res="day",
    vg_model="Sph",
    LCZinterp=True,        # use LCZ class as External Drift Kriging term
    n_jobs=-1,
    isave=True,
    lang="en",
    start="2019-07-01", end="2019-07-01",
)
print("Interpolated raster:", interp_path)

NameError: name 'lcz_interp_map' is not defined

In [19]:
# by="season" — one kriged band per season instead of per time step (requires pykrige)
interp_season_path = lcz_interp_map(
    map_path, df,
    var="airT", station_id="station",
    sp_res=100.0,
    tp_res="day",
    vg_model="Sph",
    LCZinterp=True,
    n_jobs=-1,
    isave=True,
    lang="en",
    year=2019, month=[8], day=[1] 
)
print("Seasonal interpolation:", interp_season_path)

17:08:44 - lcz_interp_map - INFO - Output saved to {path /Users/co2map/Documents/lcz4r_python/LCZ4r_output/lcz4r_interp_map.tif}


Seasonal interpolation: /var/folders/x0/lh_r13_n6_gd2h16vhk2s_3r0000gn/T/tmpeasc1owj.tif


## 7 — Plot interpolated map  `lcz_plot_interp`

Visualises the multi-band GeoTIFF from §6. A dropdown menu lets you switch
between time steps.

In [22]:
fig_interp = lcz_plot_interp(
    interp_path,
    palette="high",
    direction=1,
    title="Berlin — Interpolated Air Temperature (July 2019)",
    caption="Spherical variogram, 500 m grid, LCZ External Drift Kriging",
    lang="en"
)
fig_interp.show()

In [23]:
# Reversed 'muted' palette
lcz_plot_interp(interp_path, palette="muted", direction=-1, lang="en").show()

## 8 — Interpolation cross-validation  `lcz_interp_eval`

Evaluates interpolation accuracy using leave-one-station-out cross-validation
(LOOCV). For each time step, each station is withheld and predicted from
all remaining stations.  
Returns a table with RMSE and MAE per time step.

In [24]:
eval_df = lcz_interp_eval(
    map_path, df,
    var="airT", station_id="station",
    LOOCV=True,
    sp_res=500.0,
    tp_res="day",
    vg_model="Sph",
    LCZinterp=True,
    isave=True,
    lang="en",
    year=2019, month=7,
    start="2019-07-01", end="2019-07-07",
)
print(eval_df.shape)
eval_df.head(10)

17:09:22 - lcz_interp_eval - INFO - Output saved to {path /Users/co2map/Documents/lcz4r_python/LCZ4r_output/lcz4r_interp_eval_result.csv}


(105, 7)


station,date,observed,predicted,residual,rmse,mae
str,datetime[μs],f64,f64,f64,f64,f64
"""buch""",2019-07-01 00:00:00,22.1375,22.683199,-0.545699,0.458398,0.411129
"""dahlem""",2019-07-01 00:00:00,22.133333,22.775671,-0.642338,0.458398,0.411129
"""jagen91""",2019-07-01 00:00:00,22.151779,22.653992,-0.502213,0.458398,0.411129
"""marzahn""",2019-07-01 00:00:00,22.633333,22.600481,0.032852,0.458398,0.411129
"""albrecht""",2019-07-01 00:00:00,22.632875,22.552099,0.080776,0.458398,0.411129
"""dessauer""",2019-07-01 00:00:00,23.287058,22.777313,0.509745,0.458398,0.411129
"""bamberger""",2019-07-01 00:00:00,23.086704,22.762312,0.324392,0.458398,0.411129
"""kaniswall""",2019-07-01 00:00:00,22.703749,22.929361,-0.225613,0.458398,0.411129
"""koepenick""",2019-07-01 00:00:00,23.244954,22.747019,0.497935,0.458398,0.411129


In [ ]:
# Summary metrics per day
import polars as pl

eval_df.group_by("date").agg(
    pl.col("rmse").first(),
    pl.col("mae").first(),
    pl.col("station").count().alias("n_stations"),
).sort("date")

In [25]:
# Hold-out split variant (80 % train, 20 % test)
eval_split = lcz_interp_eval(
    map_path, df,
    var="airT", station_id="station",
    LOOCV=False,
    split_ratio=0.8,
    sp_res=500.0,
    tp_res="day",
    lang="en",
    year=2019, month=7,
    start="2019-07-01", end="2019-07-07",
)
eval_split

station,date,observed,predicted,residual,rmse,mae
str,datetime[μs],f64,f64,f64,f64,f64
"""dessauer""",2019-07-01 00:00:00,23.287058,22.813473,0.473586,0.552913,0.538315
"""bamberger""",2019-07-01 00:00:00,23.086704,22.71155,0.375154,0.552913,0.538315
"""spandauer""",2019-07-01 00:00:00,22.933167,22.367849,0.565317,0.552913,0.538315
"""airporttxl""",2019-07-01 00:00:00,22.170833,22.691423,-0.52059,0.552913,0.538315
"""rothenburg""",2019-07-01 00:00:00,23.014363,22.257433,0.75693,0.552913,0.538315
…,…,…,…,…,…,…
"""dahlemerfeld""",2019-07-06 00:00:00,17.071829,17.130867,-0.059038,0.149975,0.106024
"""jagen91""",2019-07-07 00:00:00,11.8249,12.100847,-0.275947,0.319156,0.242025
"""spandauer""",2019-07-07 00:00:00,12.1147,12.236646,-0.121946,0.319156,0.242025


## 9 — Spatial anomaly map  `lcz_anomaly_map`

Like `lcz_interp_map` but interpolates the **anomaly** (station value minus
overall spatial mean) rather than raw temperatures.  
Useful for visualising localised heat/cool pockets relative to the city average.

In [26]:
anom_path = lcz_anomaly_map(
    map_path, df,
    var="airT", station_id="station",
    sp_res=500.0,
    tp_res="day",
    vg_model="Sph",
    LCZinterp=True,
    n_jobs=-1,
    isave=True,
    lang="en",
    year=2019, month=7,
    start="2019-07-01", end="2019-07-07",
)
print("Anomaly raster:", anom_path)

17:09:39 - lcz_anomaly_map - INFO - Output saved to {path /Users/co2map/Documents/lcz4r_python/LCZ4r_output/lcz4r_anomaly_map.tif}


Anomaly raster: /var/folders/x0/lh_r13_n6_gd2h16vhk2s_3r0000gn/T/tmpte3lc60c.tif


In [27]:
# by="season" — within-season anomaly map, one band per season (requires pykrige)
anom_season_path = lcz_anomaly_map(
    map_path, df,
    var="airT", station_id="station",
    sp_res=500.0,
    tp_res="day",
    by="season",
    vg_model="Sph",
    LCZinterp=True,
    n_jobs=-1,
    isave=True,
    lang="en",
    year=2019,
)
print("Seasonal anomaly map:", anom_season_path)

17:09:43 - lcz_anomaly_map - INFO - Output saved to {path /Users/co2map/Documents/lcz4r_python/LCZ4r_output/lcz4r_anomaly_map.tif}


Seasonal anomaly map: /var/folders/x0/lh_r13_n6_gd2h16vhk2s_3r0000gn/T/tmp9er9u6ao.tif


In [28]:
# Visualise with a diverging palette (direction=-1 → cool=blue, warm=red)
fig_anom_map = lcz_plot_interp(
    anom_path,
    palette="muted",
    direction=-1,
    title="Berlin — Temperature Anomaly Map (July 2019)",
    caption="Anomaly = station value − spatial mean; 500 m grid",
    lang="en",
)
fig_anom_map.show()

## 10 — Urban Canopy Parameters  `lcz_get_ucp`

Downloads global geospatial covariates (elevation, building morphology, vegetation
cover, GHSL built-up/population) clipped to the Berlin LCZ map extent, and extracts
their values at each station location.

> **Requires**: `xarray`, `rioxarray`, `diskcache`.
> Results are cached to `./lcz4r_cache/` — the first download of a variable can take
> a few minutes (global source rasters), subsequent runs hit the cache and return in
> under a second. This demo fetches one WUMPOD variable (`elevation`) plus all 4 GHSL
> types; see §10b below to fetch the full 33-variable + GHSL suite.

In [ ]:
# One row per station — lat/lon columns are auto-detected from the DataFrame
stations_ucp = df.drop_duplicates(subset=["station"])[["station", "Latitude", "Longitude"]]

ucp_result = lcz_get_ucp(
    lcz_map=map_path,
    stations=stations_ucp,
    variables=None,       # single WUMPOD variable — fast demo
    process_ghsl=True,
    process_wumpod= True,
    process_vegetation=True,
    process_directional=True,
    n_workers=1,
)
print(ucp_result["summary"].to_dict())
ucp_result["df_vars"]

NameError: name 'df' is not defined

### 10b — Download and test ALL variables

`variables=None` (the default — just omit the argument) processes every WUMPOD (9),
vegetation (2), and directional (22) variable = 33 rasters, and `process_ghsl=True`
adds the 4 GHSL types on top = **37 rasters total**, each clipped to the Berlin LCZ
extent and cached to `./lcz4r_cache/`.

> First run downloads everything not already cached — expect it to take a while
> (each source is a separate remote fetch; the GHSL run above alone took ~3 min for
> just 4 tiles). `fail_fast=False` (default) means one failed variable doesn't stop
> the rest — check `ucp_result["failed_variables"]` afterwards.

In [30]:
stations_ucp = df.drop_duplicates(subset=["station"])[["station", "Latitude", "Longitude"]]


In [31]:
ucp_result_all = lcz_get_ucp(
    lcz_map=map_path,
    stations=stations_ucp,
    variables=None,                 # all 33 WUMPOD + vegetation + directional variables
    process_ghsl=True,              # + all 4 GHSL types
    process_wumpod=True,
    process_vegetation=True,
    process_directional=True,
    n_workers=3,
    fail_fast=False,
)
print(ucp_result_all["summary"].to_dict())
print("Failed:", ucp_result_all["failed_variables"])
ucp_result_all["df_vars"]

17:10:38 - lcz_get_ucp - INFO - ============================================================
17:10:38 - lcz_get_ucp - INFO - Urban Parameter Processor Initialized
17:10:38 - lcz_get_ucp - INFO - ============================================================
17:10:38 - lcz_get_ucp - INFO - Study Area ID: 13.087_52.338_13.762_52.676
17:10:38 - lcz_get_ucp - INFO - Target CRS: EPSG:4326
17:10:38 - lcz_get_ucp - INFO - Target Shape: (251, 501)
17:10:38 - lcz_get_ucp - INFO - Workers: 3
17:10:38 - lcz_get_ucp - INFO - Cache: lcz4r_cache
17:10:38 - lcz_get_ucp - INFO - ============================================================
17:10:38 - lcz_get_ucp - INFO - 
17:10:38 - lcz_get_ucp - INFO - Processing 33 variables
17:10:38 - lcz_get_ucp - INFO - ============================================================
Variables:   0%|          | 0/33 [00:00<?, ?var/s]17:10:38 - lcz_get_ucp - INFO -   ✓ Using cached: frc_esa_13.087_52.338_13.762_52.676.tif
17:10:38 - lcz_get_ucp - INFO -   ✓ Using cached:

{'total_variables': 37, 'successful': 36, 'failed': 1, 'total_time': 2.3221890926361084, 'success_rate': 97.2972972972973, 'failed_variables': [('hi_90', 'vsicurl failed for hi_90')]}
Failed: [('hi_90', 'vsicurl failed for hi_90')]


,station,frc_esa,elevation,hgt,lb,lc,lf,lp,urban_frc,cglc,...,zor_90,lf_135,zdm_135,zdr_135,zom_135,zor_135,built_sur,built_hei,built_vol,pop
0,airportthf,0.248090,47.680133,3.741657,0.154442,0.550385,0.031777,0.049382,0.099566,12.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,41.887947,0.029073,152.681973,0.482352
1,airporttxl,0.303181,32.832049,5.597774,0.218543,1.158112,0.048651,0.060937,0.262588,7.727226,...,0.0,0.0,0.0,0.0,0.0,0.0,158.007924,0.091436,479.131684,1.603738
2,albrecht,0.252165,45.208304,8.217909,0.565740,1.439954,0.132073,0.125841,0.092708,8.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,521.039287,0.929140,4881.972536,16.770408
3,bamberger,0.730218,41.768596,19.327860,2.141142,2.829557,0.508262,0.311389,0.731304,52.202859,...,0.0,0.0,0.0,0.0,0.0,0.0,2073.847138,6.856746,35985.674752,122.533930
4,baruth,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,berge,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,brandenburg,0.307149,43.296792,2.500000,0.016786,1.010913,0.003464,0.005876,0.214571,12.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,284.831508,0.035549,712.368680,0.190659
7,buch,0.410738,61.914787,7.392239,0.394230,1.262355,0.077786,0.131953,0.291486,56.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,1145.092549,1.699710,8893.321288,25.743249
8,dahlem,0.233801,53.447058,9.726663,0.523306,1.391703,0.114636,0.131520,0.046332,26.624524,...,0.0,0.0,0.0,0.0,0.0,0.0,460.728151,0.480248,2525.090024,8.571834
9,dahlemerfeld,0.000000,64.010918,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,7.431216,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
